# Compute precision: the dtype an encoder runs at

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.54.0/cookbook/notebooks/book/precision/compute-precision.ipynb)

Built from [`cookbook/book/chapters/precision/compute-precision.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/precision/compute-precision.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.54.0 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed samples with a compact text encoder: seconds
# to minutes, or for a chapter that fine-tunes an encoder, minutes on a GPU and up
# to hours on a CPU. The models it runs are downloaded from the Hugging Face Hub on
# first use. SCALE = "full" runs it over the published data and a larger text
# encoder instead, on the GPU: the chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.54.0", engine + "==0.54.0", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.54.0#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

**Verbs:** `[gpu] compute_precision` (`f32` / `f16` / `bf16`), set through
`jammi.connect(..., config=...)` · `generate_embeddings` · `search` ·
`describe_table` · **Background:** `bfloat16` keeps `f32`'s 8-bit exponent in half
the bytes and trims only the mantissa, where `f16` narrows the exponent to 5 bits
(Kalamkar et al. 2019) · **Checked:** live (the same encoder over the same corpus
at each precision your device admits; the vectors, the rankings and the table
identities compared).

## Two precision knobs, two stages

[The storage-precision chapters](https://f-inverse.github.io/jammi-ai/cookbook/chapters/precision/precision.html) quantize what a search index
*stores*. `compute_precision` is a different stage: the floating-point type the
encoder's weights and activations run at while it *produces* each vector. The two
compose freely: `f16` inference can write into an `int8` index, `f32` inference
into a `binary` one.

`compute_precision` takes three values. `f32` is the default and runs anywhere.
`f16` halves the memory and runs on the CPU and every supported GPU. `bf16` runs
only on a GPU with tensor-core `bf16` support (compute capability 8.0, Ampere, or
newer), and anywhere else the engine refuses it rather than quietly running
another precision.

It is deployment configuration, so it lives in the `[gpu]` section of the
engine's TOML, read when a session opens:

In [ ]:
import math
import tempfile

import jammi
from jammi_cookbook import fixtures
from jammi_cookbook.claims import claim

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
QUERIES = ["quantum computing applications", "gene editing therapies",
           "machine learning for science", "renewable energy storage",
           "catalysis and chemical synthesis"]


def run_at(precision: str):
    """Embed the corpus at `precision`; return the vectors, each query's top 3
    and the table's identity, or the refusal the engine raised."""
    config = f"{tempfile.mkdtemp()}/jammi.toml"
    with open(config, "w") as f:
        f.write(f'[gpu]\ncompute_precision = "{precision}"\n')
    with jammi.connect(f"file://{tempfile.mkdtemp()}", config=config) as db:
        db.add_source("corpus", url=fixtures.url("tiny_corpus.parquet"), format="parquet")
        try:
            table = db.generate_embeddings(source="corpus", model=MODEL,
                                           columns=["content"], key="id")
        except jammi.errors.InvalidArgument as refusal:
            return refusal
        vectors = {r["_row_id"]: r["vector"]
                   for r in db.sql(f'SELECT _row_id, vector FROM "jammi.{table}"').to_pylist()}
        tops = [db.search("corpus", query=db.encode_query(model=MODEL, query=q), k=3)
                .column("_row_id").to_pylist() for q in QUERIES]
        return vectors, tops, db.describe_table(table)["definition_hash"]


runs = {p: run_at(p) for p in ("f32", "f16", "bf16")}
for precision, run in runs.items():
    if isinstance(run, Exception):
        print(f"{precision:>5}: refused — {run}")
    else:
        print(f"{precision:>5}: {len(run[0])} vectors, identity {run[2][:12]}…")

## `f16` produces the same vectors, to four decimal places

Every vector is compared with its `f32` twin by cosine — the measure a search ranks
by — and every query's top three is compared with the `f32` run's:

In [ ]:
def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / math.sqrt(sum(x * x for x in a) * sum(y * y for y in b))


f32_vectors, f32_tops, f32_identity = runs["f32"]
f16_vectors, f16_tops, f16_identity = runs["f16"]
agreement = min(cosine(f32_vectors[k], f16_vectors[k]) for k in f32_vectors)
print(f"lowest cosine between an f16 vector and its f32 twin: {agreement:.6f}")
for q, a, b in zip(QUERIES, f32_tops, f16_tops):
    print(f"  {q:<36} f32 {a}   f16 {b}")

claim("f16 inference points every vector where f32 does", agreement > 0.999,
      f"lowest cosine {agreement:.6f}")
claim("f16 and f32 rank every query's nearest papers alike", f16_tops == f32_tops)

A half-precision forward pass perturbs each activation by a small relative error,
and cosine is blind to small changes in length: the direction, which is all a
search reads, barely moves. What `f16` buys is half the encoder's memory, and on a
GPU's tensor cores, faster inference.

## `bf16`: admitted on Ampere or newer, refused everywhere else

`bf16` keeps `f32`'s exponent range, so it is the reduced precision of choice on a
GPU that has it. On any other device the engine's model loader refuses it with a
typed error that names the requirement and the alternative. Which of the two you
see depends on the machine this runs on:

In [ ]:
bf16 = runs["bf16"]
if isinstance(bf16, Exception):
    print(f"refused: {bf16}")
    claim("bf16 is refused with the requirement and the remedy named",
          all(t in str(bf16) for t in ("bf16", "compute capability >= 8.0", "f16")))
else:
    bf16_agreement = min(cosine(f32_vectors[k], bf16[0][k]) for k in f32_vectors)
    print(f"lowest cosine between a bf16 vector and its f32 twin: {bf16_agreement:.6f}")
    claim("bf16 inference points every vector where f32 does", bf16_agreement > 0.99,
          f"lowest cosine {bf16_agreement:.6f}")

## The precision is part of the table's identity

Every result table records the definition that produced it, hashed; `describe_table`
reads it back. The same model over the same rows at two precisions is two
definitions, so the two tables never stand in for each other — a lineage check (see
[the recompute chapter](https://f-inverse.github.io/jammi-ai/cookbook/chapters/recompute/recompute.html)) knows which precision produced
the vectors it is looking at.

In [ ]:
print(f"f32 identity {f32_identity}")
print(f"f16 identity {f16_identity}")
claim("a table embedded at another precision is another definition",
      f32_identity != f16_identity)

## References

- Kalamkar, Dhiraj, Mudigere, Dheevatsa, Mellempudi, Naveen, Das, Dipankar, Banerjee, Kunal, Avancha, Sasikanth, Vooturi, Dharma Teja, Jammalamadaka, Nataraj, Huang, Jianyu, Yuen, Hector, Yang, Jiyan, Park, Jongsoo, Heinecke, Alexander, Georganas, Evangelos, Srinivasan, Sudarshan, Kundu, Abhisek, Smelyanskiy, Misha, Kaul, Bharat, Dubey, Pradeep (2019) *A Study of BFLOAT16 for Deep Learning Training* arXiv preprint arXiv:1905.12322.